# 12 - Selection set: unseen field photos for calibration and the threshold

**In short:** field photos from four sources the model has never seen are downloaded,
checked and saved as `selection.csv`. Notebook 13 uses them to choose the calibration
method (D6) and the referral threshold (D7), and to decide between ResNet-50 and the
ensemble (D2). The validation photos cannot do this: they come from the training
sources and are 99.5% right, so they contain almost no mistakes to learn from.
PlantDoc stays untouched for the final test.

| Source | Crop | Classes used | Licence |
|---|---|---|---|
| Seasonal corn (Natore, Bangladesh) | maize | common rust, gray leaf spot, healthy | CC BY 4.0 |
| MLD (Jhenaidah, Bangladesh) | maize | gray leaf spot, healthy ("Blight" left out: ambiguous) | CC BY 4.0 |
| Tomato leaf disease (Bangladesh), V1 | tomato | bacterial spot, early blight, late blight, healthy | CC BY 4.0 |
| Holeta (Ethiopia) | potato | late blight, healthy | CC BY 4.0 |

**Checks:** the licence is read from Mendeley for every source; files are downloaded
through Mendeley's API and verified against its SHA-256 checksums. The tomato set's
files are not offered by the API, so its archive is downloaded from the dataset page
and placed on Drive by hand; its checksum is recorded and verified, and the table below
says so. Only unambiguous folders are mapped. Photos that match a training-source or
PlantDoc photo (exactly, or as a near-duplicate, also when turned or mirrored) are
dropped, and near-duplicates inside the selection set are reduced to one.

**Second run (17 Oct):** the first run left out 117 tomato bacterial spot photos because
the dataset spells the folder "Bacteiral Spot"; that spelling is now mapped. The tomato
checksum from the first run is now recorded and verified. Turned copies (90, 180 and
270 degrees) are now caught as well as mirrored ones, because 41% of the Natore maize
photos were near-copies of each other. Photos saved by the first run are replaced, and
are now saved shrunk to a shorter side of 256 pixels (the size every split is shrunk to
before the model sees it), because the full-size photos filled the 15 GB Drive.

**Before running:** put the tomato archive (downloaded from
https://data.mendeley.com/datasets/jttrv2w27r/1) alone in
`MyDrive/CropDiseaseDiagnosis/datasets/raw/selection/tomato_bd/`.

No GPU needed. About 30-45 minutes: downloading the 4.6 GB maize archive, then the
duplicate checks (which show a progress bar).

In [1]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

Mounted at /content/drive


In [2]:
# Keep the Drive copy of the repository in sync with GitHub.
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

Already up to date.
Refresh index: 100% (216/216), done.
## main...origin/main
 M ml/notebooks/12_selection_set_acquisition.ipynb
?? ml/results/fusion_quick_test/
?? ml/results/selection_set/


In [3]:
%pip install -q -r {ML_DIR}/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.0 MB/s eta 0:00:00


## Licence, download and checksums

In [4]:
import importlib

import pandas as pd

from src.data import cleaning, selection_set
importlib.reload(cleaning); importlib.reload(selection_set)

SELECTION_DIR = config.RAW_DATA_DIR / "selection"   # kept photos, on Drive
LOCAL = Path("/content/selection")                  # archives and unpacked files, Colab disk only

records = []
for name, source in selection_set.SOURCES.items():
    licence = selection_set.check_licence(source)
    # The hand-downloaded archive lives on Drive; API downloads go to Colab's disk.
    dest = SELECTION_DIR / name if source.manual_sha256 is not None or name == "tomato_bd" else LOCAL / "archives" / name
    for record in selection_set.download(name, dest):
        records.append({**record, "licence": licence, "dataset": f"{source.dataset_id} v{source.version}",
                        "citation": source.citation})
sources_table = pd.DataFrame(records)
display(sources_table[["source", "dataset", "licence", "file", "bytes", "sha256", "how"]])

,source,dataset,licence,file,bytes,sha256,how
0,maize_natore,vy629dngm8 v1,CC BY 4.0,Corn leaf disease dataset.zip,4599036050,575628df92e69c169fa82c8506253d7d5886a8931605bf...,"Mendeley API, checksum verified"
1,maize_jhenaidah,3myfctrgk3 v1,CC BY 4.0,Maize dataset.zip,27340926,f15846ec615e6239ea9e282beb0b5506edfec1fb19975d...,"Mendeley API, checksum verified"
2,tomato_bd,jttrv2w27r v1,CC BY 4.0,Tomato Leaf Disease Classification Dataset.zip,285706439,7e24afdb81e0b6b0c189c5588fb57571ebe518b873dc5c...,"hand-downloaded (API offers no files), checksu..."
3,potato_holeta,v4w72bsts5 v2,CC BY 4.0,Local.rar,5334258,6928a86923c15059a838003b1113ac7fbac8c0790e6419...,"Mendeley API, checksum verified"


## Unpack and map the folders

Check this table before going on: every folder, how many photos, and what it became.
Anything "unmapped" or "excluded" is left out of the selection set.

In [5]:
tables = []
for record in records:
    archive_dir = (SELECTION_DIR if record["how"].startswith("hand") else LOCAL / "archives") / record["source"]
    unpacked = LOCAL / "unpacked" / record["source"]
    selection_set.extract(archive_dir / record["file"], unpacked)
for name in selection_set.SOURCES:
    tables.append(selection_set.inventory(name, LOCAL / "unpacked" / name))
inventory = pd.concat(tables, ignore_index=True)
summary = selection_set.folder_summary(inventory)
pd.set_option("display.max_colwidth", 80)
display(summary)

,source,folder,label_reason,images,mapped_to
0,maize_jhenaidah,Maize dataset/Fall_Armyworm,"excluded: a pest, not one of our classes",525,None
1,maize_jhenaidah,Maize dataset/Gray_Spot,mapped,555,Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
2,maize_jhenaidah,Maize dataset/Healthy,mapped,427,Corn_(maize)___healthy
3,maize_jhenaidah,Maize dataset/Leaf_Blight,excluded: ambiguous: northern or southern leaf blight,489,None
4,maize_natore,Final corn dataset/Bacterial Leaf Streak,excluded: not one of our classes,190,None
5,maize_natore,Final corn dataset/Common_rust,mapped,129,Corn_(maize)___Common_rust_
6,maize_natore,Final corn dataset/Gray_leaf_spot,mapped,1497,Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
7,maize_natore,Final corn dataset/Healthy,mapped,1038,Corn_(maize)___healthy
8,maize_natore,Final corn dataset/Maize Chlorotic Mottle Virus,excluded: not one of our classes,89,None
9,potato_holeta,Local/Healthy,mapped,363,Potato___healthy


## Duplicate checks

Fingerprints of every existing photo come from notebook 06 (`clean_manifest.csv`).

In [6]:
existing = pd.read_csv(config.METADATA_DIR / "clean_manifest.csv")
mapped = inventory[inventory["class"].notna()].reset_index(drop=True)
prints = selection_set.fingerprint_all(list(mapped["path"]))
checked = pd.concat([mapped, prints], axis=1)
unmapped = inventory[inventory["class"].isna()].assign(error=None, md5=None, phash=None, phash_variants=None,
                                                       width=None, height=None)
result = selection_set.deduplicate(pd.concat([checked, unmapped], ignore_index=True), existing,
                                   config.MIN_IMAGE_SIZE, config.NEAR_DUPLICATE_MAX_DISTANCE)
reasons = result.assign(reason=result["reason"].replace("", "kept")).groupby(["source", "reason"]).size().rename("images")
display(reasons.reset_index())
kept = result[result["keep"]]
counts = kept.pivot_table(index="class", columns="source", values="path", aggfunc="count", fill_value=0)
counts["total"] = counts.sum(axis=1)
display(counts)

Fingerprinting:   0%|          | 0/5209 [00:00<?, ?it/s]

,source,reason,images
0,maize_jhenaidah,"excluded: a pest, not one of our classes",525
1,maize_jhenaidah,excluded: ambiguous: northern or southern leaf blight,489
2,maize_jhenaidah,kept,913
3,maize_jhenaidah,near_duplicate_within_selection,69
4,maize_natore,excluded: not one of our classes,279
5,maize_natore,kept,1554
6,maize_natore,near_duplicate_within_selection,1095
7,maize_natore,near_duplicates_with_conflicting_labels,15
8,potato_holeta,kept,411
9,potato_holeta,near_duplicate_within_selection,19


source,maize_jhenaidah,maize_natore,potato_holeta,tomato_bd,total
class,,,,,
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot,554,826,0,0,1380
Corn_(maize)___Common_rust_,0,126,0,0,126
Corn_(maize)___healthy,359,602,0,0,961
Potato___Late_blight,0,0,67,0,67
Potato___healthy,0,0,344,0,344
Tomato___Bacterial_spot,0,0,0,97,97
Tomato___Early_blight,0,0,0,447,447
Tomato___Late_blight,0,0,0,92,92
Tomato___healthy,0,0,0,307,307


## Save the kept photos and `selection.csv`

Photos are saved to Drive, shrunk, as `raw/selection/<source>/<class>/<file>`; `selection.csv` has the
same columns as the other split files, so later notebooks load it the same way. `dup_group`
is kept for grouped folds.

In [ ]:
import shutil

# A rerun keeps a different set of photos, so remove the previous copies first.
# Only the class folders go; the hand-downloaded tomato archive next to them stays.
for name in selection_set.SOURCES:
    if (SELECTION_DIR / name).exists():
        for folder in [p for p in (SELECTION_DIR / name).iterdir() if p.is_dir()]:
            shutil.rmtree(folder)

rows, jobs = [], []
for i, row in kept.reset_index(drop=True).iterrows():
    safe = f"{i:05d}_{Path(row['path']).name}".replace(" ", "_")
    target = SELECTION_DIR / row["source"] / row["class"] / safe
    jobs.append((Path(row["path"]), target))
    # md5 is of the original download, so a photo can be traced back to its source file.
    rows.append({"path": str(target.relative_to(config.RAW_DATA_DIR)), "class": row["class"], "source": row["source"],
                 "crop": row["crop"], "dup_group": int(row["dup_group"]), "md5": row["md5"]})
# Saved shrunk (shorter side 256 px, as every split is before the model sees it), so the
# set takes about 100-150 MB of Drive instead of several GB.
selection_set.save_all_small(jobs)
selection = pd.DataFrame(rows)
selection.to_csv(config.SPLITS_DIR / "selection.csv", index=False)
sources_table.to_csv(config.METADATA_DIR / "selection_sources.csv", index=False)
result.drop(columns=["path"]).assign(path=result["path"].astype(str)).to_csv(
    config.METADATA_DIR / "selection_manifest.csv", index=False)
print(len(selection), "photos saved; selection.csv written to", config.SPLITS_DIR)

Saving photos:   0%|          | 0/3821 [00:00<?, ?it/s]

## Copy the summaries into the repository

In [ ]:
REPO_RESULTS = ML_DIR / "results" / "selection_set"
REPO_RESULTS.mkdir(parents=True, exist_ok=True)
sources_table.drop(columns=[]).to_csv(REPO_RESULTS / "sources.csv", index=False)
summary.to_csv(REPO_RESULTS / "folders.csv", index=False)
reasons.reset_index().to_csv(REPO_RESULTS / "checks.csv", index=False)
counts.to_csv(REPO_RESULTS / "counts.csv")
print(sorted(p.name for p in REPO_RESULTS.iterdir()))

In [ ]:
drive.flush_and_unmount()
drive.mount("/content/drive")
print("Drive synced and remounted.")

## Save to GitHub

Save the notebook first (Ctrl+S); only the results folder is committed, to a new branch.

In [ ]:
from google.colab import userdata

from src import github_sync

shutil.copy(ML_DIR / "notebooks" / "12_selection_set_acquisition.ipynb", REPO_RESULTS / "notebook.ipynb")
print(github_sync.push(
    REPO_DIR,
    branch="colab/selection-set-v2",  # the first run's branch already exists
    message="Acquire and check the selection set of unseen field photos",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
    paths=[str(REPO_RESULTS.relative_to(REPO_DIR))],
))